# Log-Normal AFT Model

Focused AFT modeling notebook using `LogNormalAFTFitter`.

This notebook uses log-normal AFT as the primary baseline because notebook 11 showed similar discrimination to log-logistic AFT but much more stable absolute LOS error.

## Imports

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import importlib.util

import numpy as np
import pandas as pd
from lifelines import LogNormalAFTFitter
from lifelines.utils import concordance_index

from config import PROCESSED_DATA_DIR

In [2]:
required_packages = ["lifelines"]
missing_packages = [
    package for package in required_packages
    if importlib.util.find_spec(package) is None
]

if missing_packages:
    message_lines = [
        "Missing packages: " + ", ".join(missing_packages),
        "Install with: .venv/bin/python -m pip install lifelines",
    ]
    raise ImportError("\n".join(message_lines))

## Load Modeling Data

In [3]:
modeling_df = pd.read_parquet(PROCESSED_DATA_DIR / "modeling_dataset.parquet")
ready_predictor_cols = pd.read_csv(
    PROCESSED_DATA_DIR / "modeling_ready_predictor_columns.csv"
)["predictor_column"].tolist()
ready_numeric_cols = pd.read_csv(
    PROCESSED_DATA_DIR / "modeling_ready_numeric_columns.csv"
)["numeric_predictor_column"].tolist()
ready_categorical_cols = pd.read_csv(
    PROCESSED_DATA_DIR / "modeling_ready_categorical_columns.csv"
)["categorical_predictor_column"].tolist()
split_df = pd.read_csv(PROCESSED_DATA_DIR / "modeling_train_test_split.csv")

print(f"Modeling dataset: {modeling_df.shape}")
print(f"Ready predictors: {len(ready_predictor_cols)}")
print(split_df["split"].value_counts())

Modeling dataset: (94444, 457)
Ready predictors: 443
split
train    75380
test     19064
Name: count, dtype: int64


In [4]:
model_df = modeling_df[
    ["subject_id", "stay_id", "duration", "event_observed"] + ready_predictor_cols
].merge(
    split_df[["stay_id", "split"]],
    on="stay_id",
    how="inner",
)

if len(model_df) != len(modeling_df):
    raise ValueError("Split merge changed row count")

if model_df["stay_id"].duplicated().any():
    raise ValueError("Duplicate stay_id rows after split merge")

train_df = model_df[model_df["split"].eq("train")].copy()
test_df = model_df[model_df["split"].eq("test")].copy()

print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(train_df[["duration", "event_observed"]].describe())

Train rows: 75,380
Test rows: 19,064
           duration  event_observed
count  75380.000000         75380.0
mean       3.617537             1.0
std        5.373389             0.0
min        0.001250             1.0
25%        1.095943             1.0
50%        1.962668             1.0
75%        3.843134             1.0
max      226.403079             1.0


## Preprocess Predictors

In [5]:
numeric_cols = [
    col for col in ready_numeric_cols
    if col in ready_predictor_cols
]
categorical_cols = [
    col for col in ready_categorical_cols
    if col in ready_predictor_cols
]

count_cols = [col for col in numeric_cols if col.endswith("_count_24h")]
numeric_cols = [col for col in numeric_cols if col not in count_cols]

print(f"Dropped raw count predictors: {len(count_cols)}")
print(count_cols)

numeric_medians = train_df[numeric_cols].median()
train_numeric_df = train_df[numeric_cols].fillna(numeric_medians)
test_numeric_df = test_df[numeric_cols].fillna(numeric_medians)

numeric_means = train_numeric_df.mean()
numeric_stds = train_numeric_df.std().replace(0, 1)
train_numeric_df = (train_numeric_df - numeric_means) / numeric_stds
test_numeric_df = (test_numeric_df - numeric_means) / numeric_stds

train_categorical_df = train_df[categorical_cols].astype("object").fillna("Missing").astype(str)
test_categorical_df = test_df[categorical_cols].astype("object").fillna("Missing").astype(str)

train_categorical_dummies_df = pd.get_dummies(
    train_categorical_df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int,
)
test_categorical_dummies_df = pd.get_dummies(
    test_categorical_df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int,
)
test_categorical_dummies_df = test_categorical_dummies_df.reindex(
    columns=train_categorical_dummies_df.columns,
    fill_value=0,
)

print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"Encoded categorical columns: {train_categorical_dummies_df.shape[1]}")

Dropped raw count predictors: 48
['dbp_count_24h', 'heart_rate_count_24h', 'mbp_count_24h', 'resp_rate_count_24h', 'sbp_count_24h', 'spo2_count_24h', 'temperature_count_24h', 'albumin_count_24h', 'alkaline_phosphatase_count_24h', 'alt_count_24h', 'anion_gap_count_24h', 'ast_count_24h', 'bicarbonate_count_24h', 'bilirubin_total_count_24h', 'bun_count_24h', 'calcium_count_24h', 'chloride_count_24h', 'creatinine_count_24h', 'glucose_count_24h', 'hematocrit_count_24h', 'hemoglobin_count_24h', 'inr_count_24h', 'lactate_count_24h', 'magnesium_count_24h', 'pco2_count_24h', 'ph_count_24h', 'phosphate_count_24h', 'platelets_count_24h', 'po2_count_24h', 'potassium_count_24h', 'pt_count_24h', 'ptt_count_24h', 'sodium_count_24h', 'wbc_count_24h', 'input_event_count_24h', 'unique_input_item_count_24h', 'antibiotic_event_count_24h', 'anticoagulant_event_count_24h', 'blood_product_event_count_24h', 'crystalloid_event_count_24h', 'enteral_nutrition_event_count_24h', 'insulin_event_count_24h', 'oral_ga

In [6]:
train_model_df = pd.concat(
    [
        train_df[["duration", "event_observed"]].reset_index(drop=True),
        train_numeric_df.reset_index(drop=True),
        train_categorical_dummies_df.reset_index(drop=True),
    ],
    axis=1,
)

test_model_df = pd.concat(
    [
        test_df[["stay_id", "duration", "event_observed"]].reset_index(drop=True),
        test_numeric_df.reset_index(drop=True),
        test_categorical_dummies_df.reset_index(drop=True),
    ],
    axis=1,
)

constant_cols = [
    col for col in train_model_df.columns
    if col not in ["duration", "event_observed"]
    and train_model_df[col].nunique(dropna=False) <= 1
]

if constant_cols:
    train_model_df = train_model_df.drop(columns=constant_cols)
    test_model_df = test_model_df.drop(columns=constant_cols)

print(f"Training matrix: {train_model_df.shape}")
print(f"Test matrix: {test_model_df.shape}")
print(f"Dropped constant columns: {len(constant_cols)}")

Training matrix: (75380, 454)
Test matrix: (19064, 455)
Dropped constant columns: 0


## Fit Log-Normal AFT

In [7]:
lognormal_penalizer = 0.1

lognormal_aft = LogNormalAFTFitter(penalizer=lognormal_penalizer)
lognormal_aft.fit(
    train_model_df,
    duration_col="duration",
    event_col="event_observed",
    show_progress=True,
)

lognormal_aft.summary.sort_values("p").head(25)

Optimization terminated successfully.
         Current function value: 1.878618
         Iterations: 72
         Function evaluations: 84
         Gradient evaluations: 84
  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 1.8786175063382036
        x: [ 2.666e-03 -7.436e-04 ...  6.427e-01 -3.013e-01]
      nit: 72
      jac: [ 1.423e-06 -5.225e-06 ... -1.140e-06  4.599e-06]
 hess_inv: [[ 9.177e-01  9.526e-02 ... -1.133e+00  1.267e-03]
            [ 9.526e-02  1.660e+00 ... -9.898e+00 -2.117e-02]
            ...
            [-1.133e+00 -9.898e+00 ...  1.266e+02  2.449e-02]
            [ 1.267e-03 -2.117e-02 ...  2.449e-02  5.544e-01]]
     nfev: 84
     njev: 84


coef  exp(coef)  \
param  covariate                                                           
sigma_ Intercept                                    -0.301328   0.739835   
mu_    resp_rate_min_24h                            -0.087000   0.916677   
       resp_rate_std_24h                            -0.097363   0.907226   
       resp_rate_max_24h                             0.101698   1.107050   
       oral_gastric_duration_hours_24h               0.076703   1.079721   
       temperature_max_24h                           0.128225   1.136809   
       spo2_max_24h                                  0.074024   1.076833   
       Intercept                                     0.642683   1.901577   
       heart_rate_max_24h                            0.102952   1.108438   
       temperature_min_24h                          -0.097061   0.907500   
       first_careunit_Neuro Intermediate             0.457870   1.580704   
       temperature_std_24h                          -0.083169   0.920195   
       careunit_group_NEURO                          0.425971   1.531077   
       enteral_nutrition_used_24h                    0.052383   1.053779   
       spo2_std_24h                                 -0.080134   0.922993   
       dbp_std_24h                                  -0.082803   0.920532   
       dbp_max_24h                                   0.080001   1.083288   
       vasopressor_inotrope_units_total_24h         -0.048265   0.952882   
       sbp_max_24h                                   0.079656   1.082915   
       crystalloid_duration_hours_24h                0.072339   1.075020   
       sbp_min_24h                                  -0.074955   0.927785   
       spo2_min_24h                                 -0.067008   0.935187   
       first_careunit_Surgery/Vascular/Intermediate  0.851513   2.343188   
       heart_rate_observed_24h                      -0.044566   0.956412   
       dbp_min_24h                                  -0.068515   0.933779   

                                                     se(coef)  coef lower 95%  \
param  covariate                                                                
sigma_ Intercept                                     0.002567       -0.306359   
mu_    resp_rate_min_24h                             0.004414       -0.095652   
       resp_rate_std_24h                             0.004973       -0.107110   
       resp_rate_max_24h                             0.005398        0.091119   
       oral_gastric_duration_hours_24h               0.004148        0.068572   
       temperature_max_24h                           0.007036        0.114434   
       spo2_max_24h                                  0.004111        0.065968   
       Intercept                                     0.041162        0.562007   
       heart_rate_max_24h                            0.006803        0.089617   
       temperature_min_24h                           0.006567       -0.109933   
       first_careunit_Neuro Intermediate             0.031642        0.395853   
       temperature_std_24h                           0.005970       -0.094871   
       careunit_group_NEURO                          0.031269        0.364685   
       enteral_nutrition_used_24h                    0.003867        0.044803   
       spo2_std_24h                                  0.006030       -0.091952   
       dbp_std_24h                                   0.006258       -0.095070   
       dbp_max_24h                                   0.006198        0.067854   
       vasopressor_inotrope_units_total_24h          0.003746       -0.055606   
       sbp_max_24h                                   0.006182        0.067540   
       crystalloid_duration_hours_24h                0.005646        0.061272   
       sbp_min_24h                                   0.005915       -0.086548   
       spo2_min_24h                                  0.005519       -0.077825   
       first_careunit_Surgery/Vascular/Intermediate  0.07142

## Evaluate Predictions

In [8]:
train_features_df = train_model_df.drop(columns=["duration", "event_observed"])
test_features_df = test_model_df.drop(columns=["stay_id", "duration", "event_observed"])

train_pred_median = lognormal_aft.predict_median(train_features_df).astype(float).clip(lower=0)
test_pred_median = lognormal_aft.predict_median(test_features_df).astype(float).clip(lower=0)

train_error = train_pred_median - train_model_df["duration"]
test_error = test_pred_median - test_model_df["duration"]

train_c_index = concordance_index(
    train_model_df["duration"],
    train_pred_median,
    train_model_df["event_observed"],
)
test_c_index = concordance_index(
    test_model_df["duration"],
    test_pred_median,
    test_model_df["event_observed"],
)

metrics = {
    "model": "lognormal_aft",
    "penalizer": lognormal_penalizer,
    "train_c_index": train_c_index,
    "test_c_index": test_c_index,
    "train_mae_days": train_error.abs().mean(),
    "test_mae_days": test_error.abs().mean(),
    "train_median_ae_days": train_error.abs().median(),
    "test_median_ae_days": test_error.abs().median(),
    "train_rmse_days": np.sqrt(np.mean(np.square(train_error))),
    "test_rmse_days": np.sqrt(np.mean(np.square(test_error))),
    "train_median_pred_days": train_pred_median.median(),
    "test_median_pred_days": test_pred_median.median(),
}

metrics_df = pd.DataFrame([metrics])
metrics_df

,model,penalizer,train_c_index,test_c_index,train_mae_days,test_mae_days,train_median_ae_days,test_median_ae_days,train_rmse_days,test_rmse_days,train_median_pred_days,test_median_pred_days
0,lognormal_aft,0.1,0.736249,0.737534,2.104418,2.157047,0.813079,0.81884,4.976568,5.124627,2.044632,2.053978


## Prediction Tail Diagnostics

These checks are here because earlier AFT fits produced unstable extreme predictions. The median prediction can look fine even when a few rows dominate RMSE.

In [9]:
prediction_tail_df = pd.DataFrame(
    {
        "split": ["train", "test"],
        "observed_max_days": [train_model_df["duration"].max(), test_model_df["duration"].max()],
        "predicted_max_days": [train_pred_median.max(), test_pred_median.max()],
        "predicted_99th_pct_days": [train_pred_median.quantile(0.99), test_pred_median.quantile(0.99)],
        "predicted_999th_pct_days": [train_pred_median.quantile(0.999), test_pred_median.quantile(0.999)],
    }
)

prediction_tail_df

,split,observed_max_days,predicted_max_days,predicted_99th_pct_days,predicted_999th_pct_days
0,train,226.403079,109.890937,10.878016,21.517438
1,test,121.303368,46.956875,10.783367,22.008449


In [10]:
train_extreme_predictions_df = train_df[["subject_id", "stay_id", "duration"]].copy()
train_extreme_predictions_df["predicted_median_los_days"] = train_pred_median.to_numpy()
train_extreme_predictions_df["absolute_error_days"] = (
    train_extreme_predictions_df["predicted_median_los_days"]
    - train_extreme_predictions_df["duration"]
).abs()

train_extreme_predictions_df.sort_values(
    "predicted_median_los_days",
    ascending=False,
).head(20)

,subject_id,stay_id,duration,predicted_median_los_days,absolute_error_days
37914,14009087,32563184,25.115880,109.890937,84.775057
3239,10352416,35043893,11.636042,49.268476,37.632435
47212,15004061,39439761,8.580394,44.451838,35.871445
8799,10944645,36261302,13.081377,41.655827,28.574450
80306,18503414,33765664,9.630671,39.305940,29.675268
40933,14330819,35975909,43.635914,37.226457,6.409458
73055,17744715,30717105,10.076215,37.181530,27.105315
52908,15613294,33399258,6.027905,34.908909,28.881004
8005,10865559,33443524,2.637049,34.301128,31.664079
88436,19366829,38563851,29.217708,33.766806,4.549097


## Extreme Prediction Investigation

Inspect the training row with the largest predicted LOS. This helps identify whether the prediction is driven by an impossible value, a rare category, or a high-leverage combination of features.

In [11]:
extreme_train_row = train_extreme_predictions_df.sort_values(
    "predicted_median_los_days",
    ascending=False,
).iloc[0]

extreme_stay_id = extreme_train_row["stay_id"]
extreme_subject_id = extreme_train_row["subject_id"]

print(extreme_train_row)

extreme_raw_df = train_df[train_df["stay_id"].eq(extreme_stay_id)].copy()
extreme_model_position = train_df.index.get_loc(extreme_raw_df.index[0])
extreme_model_df = train_model_df.iloc[[extreme_model_position]].copy()

print()
print(f"Extreme stay_id: {extreme_stay_id}")
print(f"Extreme subject_id: {extreme_subject_id}")
print(f"Raw row shape: {extreme_raw_df.shape}")

subject_id                   1.400909e+07
stay_id                      3.256318e+07
duration                     2.511588e+01
predicted_median_los_days    1.098909e+02
absolute_error_days          8.477506e+01
Name: 37914, dtype: float64

Extreme stay_id: 32563184.0
Extreme subject_id: 14009087.0
Raw row shape: (1, 448)


In [12]:
extreme_feature_values = extreme_raw_df[
    ["subject_id", "stay_id", "duration", "event_observed"]
    + [col for col in ready_predictor_cols if col in extreme_raw_df.columns]
].T.rename(columns={extreme_raw_df.index[0]: "extreme_value"})

extreme_feature_values.head(80)

,extreme_value
subject_id,14009087
stay_id,32563184
duration,25.11588
event_observed,1
first_careunit,Surgery/Vascular/Intermediate
...,...
mbp_last_24h,175.0
resp_rate_last_24h,22.0
sbp_last_24h,111.0
spo2_last_24h,94.0


In [13]:
numeric_distribution_df = train_df[numeric_cols].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).T

extreme_numeric_values = extreme_raw_df[numeric_cols].iloc[0]

numeric_outlier_df = numeric_distribution_df.join(
    extreme_numeric_values.rename("extreme_value")
)

numeric_outlier_df["above_99th"] = numeric_outlier_df["extreme_value"] > numeric_outlier_df["99%"]
numeric_outlier_df["below_1st"] = numeric_outlier_df["extreme_value"] < numeric_outlier_df["1%"]
numeric_outlier_df["missing_extreme_value"] = numeric_outlier_df["extreme_value"].isna()

numeric_outlier_df[
    numeric_outlier_df["above_99th"]
    | numeric_outlier_df["below_1st"]
    | numeric_outlier_df["missing_extreme_value"]
].sort_values(
    ["above_99th", "below_1st", "missing_extreme_value"],
    ascending=False,
).head(80)

,count,mean,std,min,1%,5%,50%,95%,99%,max,extreme_value,above_99th,below_1st,missing_extreme_value
mbp_last_24h,74943.0,78.773161,15.872445,10.0,43.0,57.0,77.00,106.000000,122.000000,249.000000,175.000000,True,False,False
mbp_trend_24h,74943.0,-5.702400,19.644769,-189.0,-61.0,-38.0,-5.00,24.000000,41.000000,187.000000,109.000000,True,False,False
creatinine_trend_24h,71962.0,-0.021404,0.487667,-26.8,-1.7,-0.3,0.00,0.400000,1.000000,5.400000,1.100000,True,False,False
other_fluid_duration_hours_24h,75380.0,10.958803,17.652497,0.0,0.0,0.0,0.05,48.344722,72.708972,144.366667,103.434444,True,False,False
sedative_analgesic_duration_hours_24h,75380.0,5.885066,12.149755,0.0,0.0,0.0,0.00,38.305444,47.600000,93.795000,65.625556,True,False,False
sedative_analgesic_mg_total_24h,75380.0,533.870498,1343.430676,0.0,0.0,0.0,0.00,3388.439821,6494.921665,24256.830005,10009.822215,True,False,False


In [14]:
extreme_categorical_df = pd.DataFrame(
    {
        "column": categorical_cols,
        "extreme_value": [extreme_raw_df[col].iloc[0] for col in categorical_cols],
        "train_frequency": [
            train_df[col].astype("object").fillna("Missing").astype(str).value_counts().get(
                str(extreme_raw_df[col].iloc[0]),
                0,
            )
            for col in categorical_cols
        ],
    }
)
extreme_categorical_df["train_frequency_pct"] = extreme_categorical_df["train_frequency"] / len(train_df)

extreme_categorical_df.sort_values("train_frequency_pct")

,column,extreme_value,train_frequency,train_frequency_pct
0,first_careunit,Surgery/Vascular/Intermediate,111,0.001473
5,marital_status,DIVORCED,5590,0.074158
7,race_grouped,BLACK,8187,0.108610
2,admission_type,URGENT,12301,0.163187
9,admission_urgency_group,URGENT,12301,0.163187
3,admission_location,TRANSFER FROM HOSPITAL,19355,0.256766
11,careunit_group,SURGICAL,20266,0.268851
10,admission_source_group,TRANSFER,20648,0.273919
1,gender,F,33183,0.440210
8,age_group,60-79,33641,0.446285


In [15]:
feature_cols = [
    col for col in train_model_df.columns
    if col not in ["duration", "event_observed"]
]

coef_for_mu = (
    lognormal_aft.summary
    .reset_index()
    .query("param == 'mu_'")
    .set_index("covariate")["coef"]
)

extreme_encoded_values = extreme_model_df[feature_cols].iloc[0]
common_feature_cols = [col for col in feature_cols if col in coef_for_mu.index]

contribution_df = pd.DataFrame(
    {
        "feature": common_feature_cols,
        "encoded_value": extreme_encoded_values[common_feature_cols].to_numpy(),
        "coef": coef_for_mu.loc[common_feature_cols].to_numpy(),
    }
)
contribution_df["contribution_to_mu"] = contribution_df["encoded_value"] * contribution_df["coef"]
contribution_df["abs_contribution_to_mu"] = contribution_df["contribution_to_mu"].abs()

contribution_df.sort_values("abs_contribution_to_mu", ascending=False).head(40)

,feature,encoded_value,coef,contribution_to_mu,abs_contribution_to_mu
396,first_careunit_Surgery/Vascular/Intermediate,1.000000,0.851513,0.851513,0.851513
18,resp_rate_max_24h,3.021530,0.101698,0.307285,0.307285
318,other_fluid_duration_hours_24h,5.238672,0.055582,0.291175,0.291175
16,heart_rate_max_24h,2.807747,0.102952,0.289062,0.289062
379,sedative_analgesic_mg_total_24h,7.053547,0.038324,0.270323,0.270323
52,mbp_last_24h,6.080589,0.038576,0.234566,0.234566
326,enteral_nutrition_used_24h,4.294488,0.052383,0.224959,0.224959
320,sedative_analgesic_duration_hours_24h,4.917012,0.044856,0.220556,0.220556
309,total_input_duration_hours_24h,3.396543,0.062455,0.212130,0.212130
59,mbp_trend_24h,5.855596,0.032510,0.190367,0.190367


In [16]:
top_contributors = contribution_df.sort_values(
    "abs_contribution_to_mu",
    ascending=False,
).head(25)["feature"].tolist()

comparison_rows = []
for feature in top_contributors:
    train_values = train_model_df[feature]
    extreme_value = extreme_model_df[feature].iloc[0]
    comparison_rows.append(
        {
            "feature": feature,
            "extreme_encoded_value": extreme_value,
            "train_mean_encoded": train_values.mean(),
            "train_std_encoded": train_values.std(),
            "train_min_encoded": train_values.min(),
            "train_99th_encoded": train_values.quantile(0.99),
            "train_max_encoded": train_values.max(),
            "coef": coef_for_mu.get(feature, np.nan),
        }
    )

contributor_comparison_df = pd.DataFrame(comparison_rows)
contributor_comparison_df

,feature,extreme_encoded_value,train_mean_encoded,train_std_encoded,train_min_encoded,train_99th_encoded,train_max_encoded,coef
0,first_careunit_Surgery/Vascular/Intermediate,1.000000,1.472539e-03,0.038346,0.000000,0.000000,1.000000,0.851513
1,resp_rate_max_24h,3.021530,3.519722e-16,1.000000,-3.281585,3.171605,7.823904,0.101698
2,other_fluid_duration_hours_24h,5.238672,1.131137e-18,1.000000,-0.620808,3.498098,7.557450,0.055582
3,heart_rate_max_24h,2.807747,4.147503e-17,1.000000,-3.387399,2.807747,9.145859,0.102952
4,sedative_analgesic_mg_total_24h,7.053547,-6.108141e-17,1.000000,-0.397393,4.437186,17.658492,0.038324
5,mbp_last_24h,6.080589,-4.245535e-16,1.000000,-4.344673,2.731868,10.756160,0.038576
6,enteral_nutrition_used_24h,4.294488,4.713072e-18,1.000000,-0.232854,4.294488,4.294488,0.052383
7,sedative_analgesic_duration_hours_24h,4.917012,8.313859e-17,1.000000,-0.484377,3.433397,7.235532,0.044856
8,total_input_duration_hours_24h,3.396543,1.206546e-16,1.000000,-0.927091,3.502646,8.725213,0.062455
9,mbp_trend_24h,5.855596,-1.131137e-18,1.000000,-9.357944,2.384050,9.837663,0.032510


In [17]:
extreme_debug_output_dir = PROCESSED_DATA_DIR / "model_outputs"
extreme_debug_output_dir.mkdir(exist_ok=True)

numeric_outlier_df.to_csv(
    extreme_debug_output_dir / "lognormal_aft_extreme_numeric_diagnostics.csv"
)
extreme_categorical_df.to_csv(
    extreme_debug_output_dir / "lognormal_aft_extreme_categorical_diagnostics.csv",
    index=False,
)
contribution_df.sort_values(
    "abs_contribution_to_mu",
    ascending=False,
).to_csv(
    extreme_debug_output_dir / "lognormal_aft_extreme_contributions.csv",
    index=False,
)
contributor_comparison_df.to_csv(
    extreme_debug_output_dir / "lognormal_aft_extreme_top_contributor_comparison.csv",
    index=False,
)

print(f"Saved extreme prediction diagnostics for stay_id {extreme_stay_id}")

Saved extreme prediction diagnostics for stay_id 32563184.0


## Coefficient Interpretation

For AFT models, `exp(coef)` is a time ratio. Values above 1 imply longer predicted ICU LOS; values below 1 imply shorter predicted ICU LOS.

In [18]:
coef_summary_df = lognormal_aft.summary.reset_index().rename(
    columns={"covariate": "feature"}
)
coef_summary_df["abs_coef"] = coef_summary_df["coef"].abs()
coef_summary_df = coef_summary_df.sort_values("abs_coef", ascending=False)

coef_summary_df[
    ["param", "feature", "coef", "exp(coef)", "p", "abs_coef"]
].head(40)

,param,feature,coef,exp(coef),p,abs_coef
190,mu_,first_careunit_Neurology,1.773836,5.893415,1.402620e-02,1.773836
186,mu_,first_careunit_Medicine/Cardiology Intermediate,-1.276757,0.278941,8.657742e-02,1.276757
182,mu_,first_careunit_Med/Surg,-0.977395,0.376290,1.755629e-01,0.977395
193,mu_,first_careunit_Surgery/Vascular/Intermediate,0.851513,2.343188,9.043860e-33,0.851513
192,mu_,first_careunit_Surgery/Trauma,0.705904,2.025677,9.926915e-03,0.705904
452,mu_,Intercept,0.642683,1.901577,5.891477e-55,0.642683
185,mu_,first_careunit_Medicine,0.516316,1.675842,5.699913e-02,0.516316
16,mu_,admission_type_DIRECT OBSERVATION,-0.470898,0.624441,3.868306e-15,0.470898
187,mu_,first_careunit_Neuro Intermediate,0.457870,1.580704,1.868943e-47,0.457870
130,mu_,careunit_group_NEURO,0.425971,1.531077,2.930945e-42,0.425971


## Test Predictions and LOS Groups

In [19]:
test_predictions_df = test_df[
    ["subject_id", "stay_id", "duration", "event_observed"]
].copy()
test_predictions_df["predicted_median_los_days"] = test_pred_median.to_numpy()
test_predictions_df["predicted_los_percentile"] = test_predictions_df[
    "predicted_median_los_days"
].rank(pct=True)

test_predictions_df.head()

,subject_id,stay_id,duration,event_observed,predicted_median_los_days,predicted_los_percentile
2,10000980,39765666,0.497535,1,0.524277,0.025126
3,10001217,37067082,1.118032,1,1.809973,0.398710
4,10001217,34592300,0.948113,1,1.310557,0.171160
6,10001843,39698942,0.825266,1,1.094421,0.093947
8,10002013,39060235,1.314352,1,1.558250,0.282260


In [20]:
los_group_df = test_predictions_df.copy()
los_group_df["predicted_los_group"] = pd.qcut(
    los_group_df["predicted_median_los_days"],
    q=5,
    labels=["shortest", "short", "middle", "long", "longest"],
    duplicates="drop",
)

los_group_summary_df = (
    los_group_df
    .groupby("predicted_los_group", observed=True)
    .agg(
        rows=("stay_id", "size"),
        median_observed_los=("duration", "median"),
        mean_observed_los=("duration", "mean"),
        median_predicted_los=("predicted_median_los_days", "median"),
        mean_predicted_los=("predicted_median_los_days", "mean"),
    )
    .reset_index()
)

los_group_summary_df

,predicted_los_group,rows,median_observed_los,mean_observed_los,median_predicted_los,mean_predicted_los
0,shortest,3813,0.828877,1.234372,1.114384,1.008667
1,short,3813,1.421725,2.158713,1.600140,1.597945
2,middle,3812,1.914977,2.965706,2.053978,2.063436
3,long,3813,2.782708,4.311872,2.774955,2.806918
4,longest,3813,4.938345,7.726167,4.689190,5.643838


## Save Outputs

In [21]:
model_output_dir = PROCESSED_DATA_DIR / "model_outputs"
model_output_dir.mkdir(exist_ok=True)

metrics_df.to_csv(model_output_dir / "lognormal_aft_metrics.csv", index=False)
pd.Series(count_cols, name="dropped_count_predictor").to_csv(
    model_output_dir / "lognormal_aft_dropped_count_predictors.csv",
    index=False,
)
prediction_tail_df.to_csv(model_output_dir / "lognormal_aft_prediction_tail.csv", index=False)
coef_summary_df.to_csv(model_output_dir / "lognormal_aft_coefficients.csv", index=False)
test_predictions_df.to_csv(model_output_dir / "lognormal_aft_test_predictions.csv", index=False)
los_group_summary_df.to_csv(
    model_output_dir / "lognormal_aft_los_group_summary.csv",
    index=False,
)
train_extreme_predictions_df.sort_values(
    "predicted_median_los_days",
    ascending=False,
).head(100).to_csv(
    model_output_dir / "lognormal_aft_top_train_predictions.csv",
    index=False,
)

print(f"Saved log-normal AFT outputs to {model_output_dir}")

Saved log-normal AFT outputs to /Users/brandonwu/Documents/ICU_LOS_Survival_Analysis/data/processed/model_outputs


## Modeling Notes

In [22]:
print("Modeling notes:")
print("- Log-normal AFT is the focused baseline because it had the most stable absolute LOS metrics.")
print("- Raw *_count_24h predictors are excluded because charting density created high-leverage outliers.")
print("- Use C-index for ranking performance and MAE/RMSE for absolute LOS prediction error.")
print("- Check prediction-tail outputs before trusting long-stay predictions.")
print("- Coefficients are time ratios: exp(coef) > 1 means longer predicted LOS.")

Modeling notes:
- Log-normal AFT is the focused baseline because it had the most stable absolute LOS metrics.
- Raw *_count_24h predictors are excluded because charting density created high-leverage outliers.
- Use C-index for ranking performance and MAE/RMSE for absolute LOS prediction error.
- Check prediction-tail outputs before trusting long-stay predictions.
- Coefficients are time ratios: exp(coef) > 1 means longer predicted LOS.
